# <font size=7>**Proposal Automation System**</font>

# <font size=7>**LangGraph + RAG + Mistral + OpenAI**</font>

### <font size=5>**Lu Zhu**</font>

This project implements an automated, citation-aware **NIH proposal generation system** powered by a hybrid RAG pipeline and a **multi-agent LangGraph workflow**. The system uses **two different LLM APIs for different roles**:

* **Mistral** – creative generation (research ideas, proposal drafting, rewriting)
* **OpenAI GPT-4.1** – authoritative NIH-style reviewing and scoring

This separation provides both **creativity** (Mistral) and **rigor** (OpenAI), which significantly improves proposal quality.

---

## **Key Pipeline Innovations**

#### **1. Accurate Reference Integration (LLM + Semantic Scholar + Manual Additions)**

* LLM extracts **only the real title** from each PDF.
* Titles are matched against a curated **Semantic Scholar citation list**.
* Any unmatched items are logged for manual addition and correction.
* Final citations skip empty fields and output only valid metadata.

#### **2. Idea-Driven RAG Refresh**

After generating multiple candidate ideas:

1. **Mistral generates 10 research ideas.**
2. **OpenAI selects the strongest idea.**
3. **A new RAG search is performed specifically for that selected idea.**
4. All subsequent sections retrieve evidence **only related to this chosen research direction**, ensuring tight thematic alignment.

This avoids the common problem of mixing irrelevant references across sections.

#### **3. Section-Level Agentic Workflow**

Each section (Aims → Significance → Innovation → Approach) goes through a controlled cycle:

1. **RAG retrieval (idea-focused)**
2. **Mistral generates draft**
3. **OpenAI evaluates with NIH-style scoring & critique**
4. **Mistral revises based on reviewer feedback**
5. Section is finalized only when considered strong

This ensures every part of the proposal is **internally consistent**, **idea-aligned**, and **NIH-style compliant**.

---

## **Technical Components**

* **MiniLM-L6-v2 embeddings** 
* **FAISS HNSW index (M=32)**
* **Chunking with embedded citation metadata**
* **LangGraph orchestration** controlling all proposal states
* **Human-in-the-loop revision**
---

## **Final Output**

The system produces:

* Fully assembled **NIH-style proposal**
* Final **reference list** (cleaned, canonicalized, deduplicated)
* Full **reviewer scoring report**
* Section-level drafts and revisions

---

## **Full Workflow**

                   ┌────────────────────────────────┐
                   │      Load Research Papers      │
                   └──────────────────┬─────────────┘
                                      │
                   ┌────────────────────────────────┐
                   │ Extract Paper Metadata/Text    │
                   └──────────────────┬─────────────┘
                                      │
                   ┌────────────────────────────────┐
                   │ Embed Research Papers          │
                   │ Build FAISS index              │
                   └──────────────────┬─────────────┘
                                      │
                   ┌────────────────────────────────┐
                   │   Normalize & Store Metadata   │
                   └──────────────────┬─────────────┘
                                      │
                                      ▼
                   ┌────────────────────────────────┐
                   │        Read NOFO Document      │
                   └──────────────────┬─────────────┘
                                      │
                   ┌────────────────────────────────┐
                   │ Extract NOFO Requirements      │
                   │ (parse → structure → embed)    │
                   └──────────────────┬─────────────┘
                                      │
                   ┌────────────────────────────────┐
                   │ Filter Research Papers by Topic│
                   │ (NOFO embedding similarity)    │
                   └──────────────────┬─────────────┘
                                      │
                   ┌────────────────────────────────┐
                   │      Generate Research Ideas   │
                   │ (LLM using embedded papers)    │
                   └──────────────────┬─────────────┘
                                      │
                   ┌────────────────────────────────┐
                   │      Select Best Research Idea │
                   │ (LLM scoring vs NOFO reqs)     │
                   └──────────────────┬─────────────┘
                                      │
                                      │
                   ┌────────────────────────────────┐
                   │ Filter Research Papers by      │
                   │ selected best research idea    │
                   └──────────────────┬─────────────┘
                                      │
                                      ▼
                ╔═════════════════════════════════════════╗
                ║          LANGGRAPH PIPELINE             ║
                ║   Automated Proposal Drafting Workflow  ║
                ╚═════════════════════════════════════════╝
                                      │
                   ┌────────────────────────────────┐
                   │ retrieve_idea_specific_evidence│
                   └──────────────────┬─────────────┘
                                      │
                   ┌────────────────────────────────┐
                   │         set_next_section        │
                   └──────────────────┬─────────────┘
                                      │
                   ┌────────────────────────────────┐
                   │         generate_section        │
                   │ (RAG + NOFO + Global Citations) │
                   └──────────────────┬─────────────┘
                                      │
                   ┌────────────────────────────────┐
                   │   reviewer_evaluate_section    │
                   └──────────────────┬─────────────┘
                                      │
                   ┌────────────────────────────────┐
                   │ revise_section_after_reviewer  │
                   └──────────────────┬─────────────┘
                                      │
                   ┌────────────────────────────────┐
                   │         set_next_section        │
                   └──────────────────┬─────────────┘
                                      │
                    ←── loop until ALL proposal sections completed ──→
                                      │
                                      ▼
                   ┌────────────────────────────────┐
                   │         combine_sections       │
                   └──────────────────┬─────────────┘
                                      │
                   ┌────────────────────────────────┐
                   │    apply_final_human_revision  │
                   │ (global NIH cleanup + citations) │
                   └──────────────────┬─────────────┘
                                      │
                   ┌────────────────────────────────┐
                   │    final_reviewer_evaluation    │
                   └──────────────────┬─────────────┘
                                      │
                   ┌────────────────────────────────┐
                   │         finalize_proposal       │
                   └─────────────────────────────────┘



## STEP 1 — Import Libraries & Load API Keys

In [38]:
# ! pip install faiss-cpu
# ! pip install PyMuPDF
# ! pip install langgraph
# ! pip install openai
# !pip install langchain-mistralai
# !pip install --upgrade langchain langchain-core langchain-community langchain-mistralai
# ! pip install reportlab


In [1]:
# ================================
# Step 1: Environment Setup
# ================================

import os
import json
import logging

import numpy as np
from tqdm import tqdm
from dotenv import load_dotenv

# For embedding + FAISS
from sentence_transformers import SentenceTransformer
import faiss

# For PDF reading
import fitz  # PyMuPDF

# For LangGraph
from langgraph.graph import StateGraph, END

# For LLM calls
import requests
import openai

# ---------------------------------
# Load API Keys from config.json
# ---------------------------------

CONFIG_PATH = "./config.json"

if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError("config.json not found in project root.")

with open(CONFIG_PATH, "r") as f:
    cfg = json.load(f)

os.environ["OPENAI_API_KEY"] = cfg["OPENAI_API_KEY"]
os.environ["MISTRAL_API_KEY"] = cfg["MISTRAL_API_KEY"]

openai.api_key = cfg["OPENAI_API_KEY"]

print("✅ API keys loaded successfully.")


✅ API keys loaded successfully.


## STEP 2 — Load & Ingest All PDFs

- Scan ./papers/ folder
- Extract text from all PDFs using PyMuPDF
- Chunk text into RAG-friendly pieces
- Save everything to ./data/
- Skip work if checkpoint files already exist
- Show clear progress logs (tqdm + print statements)

This is one of the heaviest steps in the notebook, so I am designing it with robust checkpointing and progress bars to avoid reruns.

### Integrate Citation Metadata With PDF Contents
We use a hybrid metadata extraction strategy that combines LLM title parsing, Semantic Scholar metadata, and manual augmentation to ensure reliable citations for every PDF used in the RAG system.

1. LLM-Based Title Extraction

- For each PDF, we extract the first ~300–600 words.
- A compact LLM (Mistral) is prompted to return only the paper title.
- This is far more accurate than reading PDF metadata or first-line heuristics.
- The extracted title becomes the primary key for matching.

2. Semantic Scholar Master Citation List

- We downloaded structured metadata for the authors (titles, authors, year, venue, DOI) using Semantic Scholar's API.
- Stored in master_citations.json.
- Titles are normalized to compute fuzzy matches.
- This serves as the canonical reference database.

3. Manual Additions

- Some references (technical reports, under-review papers, workshop papers) are not available via API.
- We manually added these into the master list.
- This ensures full coverage across all PDFs.

4. Title Matching (LLM → Master List)

- Normalize the LLM-generated title.
- Compare against all master titles using fuzzy matching.
- If similarity ≥ 0.65 → assign full metadata (authors, year, venue, DOI).
- Otherwise → place into unmatched_citations.json for manual review.

5. Metadata Binding During Chunking

- Each PDF is split into text chunks.
- Every chunk is saved with its full citation metadata:
    - title
    - authors
    - year
    - venue
    - doi
- This allows the RAG system to produce inline citations and a clean final reference list.

6. Fail-Safe Defaults

- If no match is found:
    - title = "Unknown Title"
    - year = "n.d."
    - Other fields left blank
- These entries remain visible for later correction.

7. Why This Works Well

- LLM → best possible title extraction
- Semantic Scholar → authoritative metadata
- Manual fixes → complete coverage
- Fuzzy matching → robust alignment
- Chunk-level metadata → accurate citations in RAG outputs
- This results in a high-quality, self-improving citation system that supports NIH-style scientific writing and reproducible RAG workflows.

### Chunking Strategy

This project uses a hybrid paragraph-based + length-controlled chunking strategy:
- Split by paragraphs (double newline or large line breaks)
- Merge adjacent paragraphs until the chunk is ≥ 300 characters and ≤ 1500 characters
- Ensures natural flow and RAG coherence

In [74]:
# ================================
# Step 2: Directory Setup
# ================================

import os
import json
from glob import glob

DATA_DIR = "./data"
PAPERS_DIR = "./papers"

os.makedirs(DATA_DIR, exist_ok=True)

RAW_TEXT_PATH = f"{DATA_DIR}/papers_raw.json"
CHUNKS_PATH = f"{DATA_DIR}/paper_chunks.json"

print("Data directory ready:", DATA_DIR)
print("Papers directory:", PAPERS_DIR)


Data directory ready: ./data
Papers directory: ./papers


In [73]:
# ================================
# Step 2A: PDF Text Extraction
# ================================

import fitz  # PyMuPDF
from tqdm import tqdm

def extract_text_from_pdf(pdf_path):
    """Extracts text from a single PDF using PyMuPDF."""
    doc = fitz.open(pdf_path)
    text = []

    for page in doc:
        text.append(page.get_text())
    doc.close()

    return "\n".join(text)

def extract_full_text(pdf_path, max_pages=5):
    """Extract up to `max_pages` of text for metadata extraction."""
    doc = fitz.open(pdf_path)
    texts = []
    for i, page in enumerate(doc):
        texts.append(page.get_text())
        if i + 1 >= max_pages:
            break
    doc.close()
    return "\n".join(texts)



In [72]:
# ================================
# Step 2C: Chunking Strategy
# ================================

def chunk_text(text, min_len=300, max_len=1500):
    """Paragraph-based chunking with merge rules."""
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks = []
    buffer = ""

    for para in paragraphs:
        if len(buffer) + len(para) < max_len:
            buffer += " " + para
        else:
            if len(buffer) > min_len:
                chunks.append(buffer.strip())
            buffer = para

    if len(buffer) > min_len:
        chunks.append(buffer.strip())

    return chunks


In [71]:
# ===================================================
# Mistral API Wrapper
# ===================================================

import requests
import os

MISTRAL_API_KEY = os.environ["MISTRAL_API_KEY"]
MISTRAL_API_URL = "https://api.mistral.ai/v1/chat/completions"
MISTRAL_MODEL = "mistral-large-latest"

def mistral_chat(messages, temperature=0.2, max_tokens=None):
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {MISTRAL_API_KEY}"
    }

    payload = {
        "model": MISTRAL_MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    response = requests.post(MISTRAL_API_URL, json=payload, headers=headers)
    response.raise_for_status()
    data = response.json()
    return data["choices"][0]["message"]["content"]

In [11]:
import requests
import json
import time

AUTHOR_ID = "2638935"   # Ian McCulloh
SAVE_PATH = "./data/master_citations.json"

# Semantic Scholar Graph API endpoint
BASE_URL = "https://api.semanticscholar.org/graph/v1"

FIELDS = (
    "title,year,venue,authors,externalIds,url,abstract,publicationTypes"
)

def get_author_papers(author_id, limit=1000):
    """
    Retrieve full list of papers for an author from Semantic Scholar API.
    Handles pagination if needed.
    """

    url = f"{BASE_URL}/author/{author_id}/papers"
    params = {
        "fields": FIELDS,
        "limit": limit,
        "offset": 0
    }

    all_papers = []

    while True:
        print(f"Fetching offset {params['offset']} ...")

        resp = requests.get(url, params=params)
        data = resp.json()

        papers = data.get("data", [])
        if not papers:
            break

        all_papers.extend(papers)

        # Pagination: break if fewer than limit returned
        if len(papers) < limit:
            break

        params["offset"] += limit
        time.sleep(0.5)

    return all_papers


def normalize_paper(p):
    """
    Convert API paper object into a clean dictionary.
    """

    # Authors: "Last, F.; Last, F."
    authors = ", ".join(a.get("name", "") for a in p.get("authors", []))

    doi = p.get("externalIds", {}).get("DOI", "")
    venue = p.get("venue", "") or "Unknown Venue"

    return {
        "paper_id": p.get("paperId", ""),
        "title": p.get("title", ""),
        "year": p.get("year", ""),
        "venue": venue,
        "doi": doi,
        "authors": authors,
        "abstract": p.get("abstract", ""),
        "url": p.get("url", ""),
        "pdf_url": p.get("openAccessPdf", {}).get("url", ""),
        "publicationTypes": p.get("publicationTypes", []),
    }


# -------------------------------------------------------
# RUN EXTRACTION
# -------------------------------------------------------

print(f"Retrieving full publication list for author {AUTHOR_ID} ...")

papers_raw = get_author_papers(AUTHOR_ID)
papers_clean = [normalize_paper(p) for p in papers_raw]

# Save
with open(SAVE_PATH, "w") as f:
    json.dump(papers_clean, f, indent=2)

print(f"\n📄 Total papers retrieved: {len(papers_clean)}")
print(f"📚 Saved citation metadata to: {SAVE_PATH}")


Retrieving full publication list for author 2638935 ...
Fetching offset 0 ...

📄 Total papers retrieved: 73
📚 Saved citation metadata to: ./data/master_citations.json


In [70]:
# ============================================================
# 0. Imports
# ============================================================
import json
import re
from difflib import SequenceMatcher

# ============================================================
# Load master citations
# ============================================================

MASTER_CITATIONS_PATH = "./data/master_citations.json"

with open(MASTER_CITATIONS_PATH, "r") as f:
    MASTER_CITATIONS = json.load(f)

# Normalize and index
MASTER_INDEX = []

for x in MASTER_CITATIONS:
    # Auto-generate a paper_id if missing
    paper_id = x.get("paper_id")
    if not paper_id:
        norm_title = re.sub(r"[^a-z0-9]+", "-", x["title"].lower()).strip("-")
        paper_id = f"auto_{norm_title[:40]}"  # deterministic ID

    MASTER_INDEX.append({
        "paper_id": paper_id,
        "title": x.get("title", ""),
        "title_norm": re.sub(r"[^a-z0-9]+", " ", x.get("title", "").lower()).strip(),
        "authors": x.get("authors", ""),
        "year": x.get("year", ""),
        "venue": x.get("venue", ""),
        "doi": x.get("doi", ""),
    })


# ============================================================
# Title-only matching using LLM output
# ============================================================

def normalize(text):
    return re.sub(r"[^a-z0-9]+", " ", text.lower()).strip()

def fuzzy(a, b):
    return SequenceMatcher(None, a, b).ratio()


def match_by_title(llm_title):
    """
    Matching logic based ONLY on LLM-extracted title.
    No abstract use, no intro page matching.
    """
    llm_norm = normalize(llm_title)

    best = None
    best_score = 0

    for entry in MASTER_INDEX:
        score = fuzzy(llm_norm, entry["title_norm"])
        if score > best_score:
            best = entry
            best_score = score

    # very safe threshold
    if best_score >= 0.65:
        return best

    return None

# ============================================================
# LLM title extractor
# ============================================================

import fitz

def extract_pdf_intro(pdf_path, max_chars=2000):
    doc = fitz.open(pdf_path)
    buf = ""
    for page in doc:
        buf += page.get_text()
        if len(buf) > max_chars:
            break
    doc.close()
    return buf[:max_chars]


def llm_extract_title(intro_text, mistral_chat):
    prompt = f"""
    Extract ONLY the scientific paper title from the text below.
    Return only the title. No quotes. No extra text.

    TEXT:
    {intro_text[:600]}
    """

    resp = mistral_chat([{"role":"user","content":prompt}])
    title = resp.content.strip() if hasattr(resp,"content") else resp.strip()
    title = title.replace("\n"," ").strip()

    # fallback heuristic if LLM fails
    if len(title) < 5 or len(title.split()) < 2:
        lines = [l.strip() for l in intro_text.split("\n") if len(l.strip()) > 6]
        if lines:
            return max(lines, key=len)  # longest line = title heuristic
        return "Untitled Document"

    return title


# ======================================================
# LOGGING: Track metadata extraction quality
# ======================================================
audit_log = []     # list of dicts, one per PDF
unmatched_papers = []  # filenames not matched to master list
llm_failed = []        # ones where LLM extraction failed



## STEP 3 — Embeddings + FAISS HNSW Index

- Load MPNet model (all-mpnet-base-v2)
- Embed all 1,676 text chunks
- Build an HNSW Flat (M = 32) index
- Save the index and embedding vectors
- Log progress and skip recomputation if checkpoints already exist

In [75]:
# ======================================
# Step 3A: Load Embedding Model
# ======================================

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

EMBEDDING_DIM = embedding_model.get_sentence_embedding_dimension()

print(f"Embedding model loaded. Dimension = {EMBEDDING_DIM}")


Embedding model loaded. Dimension = 384


In [6]:
# ======================================
# Step 3B: Set Embedding Checkpoint Paths
# ======================================

EMBED_PATH = f"{DATA_DIR}/embeddings.npy"
META_PATH = f"{DATA_DIR}/metadata.json"
FAISS_PATH = f"{DATA_DIR}/hnsw_index.faiss"


In [44]:
# ===============================================
# Step 3C: Low-Memory Embedding Pipeline
# ===============================================

import numpy as np
import json
from tqdm import tqdm

BATCH_SIZE = 4   # ultra-safe batch size
USE_FP16 = True  # reduce RAM usage by 50%

if os.path.exists(EMBED_PATH) and os.path.exists(META_PATH):
    print("Loading existing embeddings and metadata...")

    embeddings = np.load(EMBED_PATH)
    with open(META_PATH, "r") as f:
        metadata = json.load(f)

    print("Embeddings loaded:", embeddings.shape)

else:
    print(f"Generating embeddings with batch size = {BATCH_SIZE}...")

    texts = [c["chunk"] for c in paper_chunks]
    total = len(texts)

    embeddings_list = []
    metadata = []

    for i in tqdm(range(0, total, BATCH_SIZE), desc="Embedding chunks"):
        batch = texts[i : i + BATCH_SIZE]
        batch_embeddings = embedding_model.encode(batch, show_progress_bar=False)

        if USE_FP16:
            batch_embeddings = batch_embeddings.astype("float16")

        embeddings_list.append(batch_embeddings)

        # Save metadata
        for j in range(len(batch)):
            chunk = paper_chunks[i + j]

            # Guarantee citation fields exist (safe patch)
            chunk_fixed = {
                "chunk": chunk["chunk"],
                "source": chunk.get("source", ""),

                "citation_id":  chunk.get("citation_id", f"auto_{i+j}"),
                "title":        chunk.get("title", "Missing Title"),
                "authors":      chunk.get("authors", "Unknown Authors"),
                "year":         chunk.get("year", "n.d."),
                "venue":        chunk.get("venue", "Unknown Venue"),
                "doi":          chunk.get("doi", ""),
            }

            metadata.append(chunk_fixed)

    # Concatenate only after all batches are processed
    embeddings = np.vstack(embeddings_list)

    np.save(EMBED_PATH, embeddings)
    with open(META_PATH, "w") as f:
        json.dump(metadata, f)

    print("Embeddings saved:", EMBED_PATH)
    print("Metadata saved:", META_PATH)

print("Total embeddings:", embeddings.shape)


Loading existing embeddings and metadata...
Embeddings loaded: (1676, 384)
Total embeddings: (1676, 384)


In [43]:
# ====================================================
# Step 3D: Build or Load FAISS HNSW Index
# ====================================================

if os.path.exists(FAISS_PATH):
    print("Loading existing HNSW index...")
    index = faiss.read_index(FAISS_PATH)
    print("Index loaded. Entries:", index.ntotal)

else:
    print("Building FAISS HNSW index (M=32, memory-optimized)...")

    M = 32
    index = faiss.IndexHNSWFlat(EMBEDDING_DIM, M)

    # Lower efConstruction reduces RAM usage (still good accuracy)
    index.hnsw.efConstruction = 40

    # Incrementally add embeddings in small batches
    for i in tqdm(range(0, embeddings.shape[0], 32), desc="Indexing embeddings"):
        batch = embeddings[i : i + 32]
        index.add(batch.astype("float32"))  # HNSW requires float32

    # Save index
    faiss.write_index(index, FAISS_PATH)
    print("HNSW index saved:", FAISS_PATH)

print("FAISS index size:", index.ntotal)


Loading existing HNSW index...
Index loaded. Entries: 1676
FAISS index size: 1676


## STEP 4 — Build Retrieval Functions

- Top-K Similarity Search
- Chunk Retrieval Wrapper
- RAG Context Assembler
- Utility Functions
- Sanity-Check Retrieval

In [7]:
# ==========================================================
# Step 4A: Load FAISS Index + Metadata
# ==========================================================

import numpy as np
import json
import faiss

# Load metadata
with open(META_PATH, "r") as f:
    metadata = json.load(f)

# Load embeddings
embeddings = np.load(EMBED_PATH)

# Load FAISS index
index = faiss.read_index(FAISS_PATH)

print("Retrieval system ready.")
print("Vectors:", embeddings.shape)
print("FAISS index entries:", index.ntotal)


Retrieval system ready.
Vectors: (1676, 384)
FAISS index entries: 1676


In [9]:
# ==========================================================
# Step 4B: Normalize Query Vectors
# ==========================================================

def embed_query(text):
    vec = embedding_model.encode([text], show_progress_bar=False)
    vec = vec.astype("float32")
    faiss.normalize_L2(vec)
    return vec


In [10]:
# ==========================================================
# Step 4C: Top-K FAISS Search
# ==========================================================

def search_chunks(query, k=6):
    """Search top-k most relevant chunks."""
    qvec = embed_query(query)
    distances, indices = index.search(qvec, k)

    results = []
    for rank, idx in enumerate(indices[0]):
        if idx == -1:
            continue
        results.append({
            "rank": rank + 1,
            "distance": float(distances[0][rank]),

            # TEXT
            "chunk": metadata[idx]["chunk"],

            # METADATA
            "source": metadata[idx]["source"],
            "citation_id": metadata[idx]["citation_id"],
            "title": metadata[idx]["title"],
            "authors": metadata[idx]["authors"],
            "year": metadata[idx]["year"],
            "venue": metadata[idx]["venue"],
            "doi": metadata[idx]["doi"],

            "id": int(idx)
        })

    return results


In [11]:
# ==========================================================
# Step 4D: Build Context for RAG
# ==========================================================

def build_context(query, k=6):
    """Retrieve top-k chunks and stitch them into a RAG context string."""
    results = search_chunks(query, k=k)
    context = "\n\n--- Retrieved Chunk ---\n\n".join([r["chunk"] for r in results])

    return context, results


In [20]:
# ==========================================================
# Step 4E: Sanity Check Retrieval
# ==========================================================

test_query = "digital mental health"
context, results = build_context(test_query, k=3)

print("🔎 Retrieval Test for:", test_query)
for r in results:
    print(f"\nRank {r['rank']} | Source: {r['source']}")
    print(r["chunk"][:400], "...")


🔎 Retrieval Test for: digital mental health

Rank 1 | Source: Social Media Mental Health Final
ASONAM ’23, November 6-9, 2023, Kusadasi, Turkey © 2023 Association for Computing Machinery. 
ACM ISBN 979-8-4007-0409-3/23/11. . . $15.00 https://doi.org/10.1145/3625007.3627490 
Fragile Minds: Exploring the Link Between Social 
Media and Young Adult Mental Health 
 
 
 
 
Ben Cohen 
Whiting School of Engineering 
Johns Hopkins University 
Baltimore, MD, USA 
bcohen49@jhu.edu 
 
Ian McCulloh 
Joh ...

Rank 2 | Source: Social Media Mental Health Final
jointly declare a “national state of emergency in children’s 
mental health” [9]. 
Much of the blame for this degradation in teenage mental 
health has been laid at the feet of social media. For example, a 
2019 study of 12,886 individuals in England ages 13–16 found 
that those who used social media services more than three times 
daily self-reported worse mental health and well-being than 
those ...

Rank 3 | Source: NAP Behavioral Sci Intel
A

## STEP 5 — Ingest the NOFO: Build the NOFO RAG index

Create a separate RAG store for NOFO requirements, and build structured fields such as:
- Significance Requirements
- Innovation Requirements
- Approach Requirements
- Responsiveness Criteria
- Review Criteria (Factor 1–3)

In [12]:
# ===========================================
# Step 5A: NOFO Paths
# ===========================================

NOFO_PATH = "./PAR-25-136_ Laboratories to Optimize Digital Health (R0….pdf"

NOFO_RAW_PATH = f"{DATA_DIR}/nofo_raw.txt"
NOFO_CHUNKS_PATH = f"{DATA_DIR}/nofo_chunks.json"
NOFO_EMBED_PATH = f"{DATA_DIR}/nofo_embeddings.npy"
NOFO_INDEX_PATH = f"{DATA_DIR}/nofo_index.faiss"

print("NOFO file:", NOFO_PATH)


NOFO file: ./PAR-25-136_ Laboratories to Optimize Digital Health (R0….pdf


In [13]:
# ===========================================
# Step 5B: Extract NOFO text
# ===========================================

if os.path.exists(NOFO_RAW_PATH):
    print("Loading existing NOFO raw text...")
    with open(NOFO_RAW_PATH, "r") as f:
        nofo_text = f.read()

else:
    print("Extracting text from NOFO PDF...")
    nofo_text = extract_text_from_pdf(NOFO_PATH)

    with open(NOFO_RAW_PATH, "w") as f:
        f.write(nofo_text)

    print("Saved NOFO raw text:", NOFO_RAW_PATH)

print("NOFO text length:", len(nofo_text))


Loading existing NOFO raw text...
NOFO text length: 79887


In [15]:
# ===========================================
# Step 5C: Chunk NOFO (or load checkpoint)
# ===========================================

if os.path.exists(NOFO_CHUNKS_PATH):
    print("Loading existing NOFO chunks...")
    with open(NOFO_CHUNKS_PATH, "r") as f:
        nofo_chunks = json.load(f)

else:
    print("Chunking NOFO text...")
    nofo_chunks = []

    chunks = chunk_text(nofo_text, min_len=200, max_len=1200)

    for i, ch in enumerate(chunks):
        nofo_chunks.append({
            "chunk": ch,
            "source": "NOFO"
        })

    with open(NOFO_CHUNKS_PATH, "w") as f:
        json.dump(nofo_chunks, f)

    print("Saved NOFO chunks:", NOFO_CHUNKS_PATH)

print("Total NOFO chunks:", len(nofo_chunks))


Loading existing NOFO chunks...
Total NOFO chunks: 11


In [16]:
# ===========================================
# Step 5D: Embed NOFO chunks (memory-optimized)
# ===========================================

if os.path.exists(NOFO_EMBED_PATH):
    print("Loading existing NOFO embeddings...")
    nofo_embeddings = np.load(NOFO_EMBED_PATH)

else:
    print("Encoding NOFO chunks...")

    nofo_embeddings_list = []

    for i in tqdm(range(0, len(nofo_chunks), 4), desc="Embedding NOFO"):
        batch = [nofo_chunks[j]["chunk"] for j in range(i, min(i+4, len(nofo_chunks)))]
        batch_emb = embedding_model.encode(batch, show_progress_bar=False)

        nofo_embeddings_list.append(batch_emb.astype("float32"))

    nofo_embeddings = np.vstack(nofo_embeddings_list)

    np.save(NOFO_EMBED_PATH, nofo_embeddings)
    print("Saved NOFO embeddings:", NOFO_EMBED_PATH)

print("NOFO embedding shape:", nofo_embeddings.shape)


Loading existing NOFO embeddings...
NOFO embedding shape: (11, 768)


In [17]:
# ===========================================
# Step 5E: Build NOFO HNSW Index
# ===========================================

if os.path.exists(NOFO_INDEX_PATH):
    print("Loading existing NOFO index...")
    nofo_index = faiss.read_index(NOFO_INDEX_PATH)

else:
    print("Building NOFO HNSW index (M=32)...")

    nofo_index = faiss.IndexHNSWFlat(EMBEDDING_DIM, 32)
    nofo_index.hnsw.efConstruction = 40

    # Add in small batches
    for i in tqdm(range(0, nofo_embeddings.shape[0], 16), desc="Indexing NOFO"):
        batch = nofo_embeddings[i : i + 16]
        nofo_index.add(batch.astype("float32"))

    faiss.write_index(nofo_index, NOFO_INDEX_PATH)
    print("Saved NOFO FAISS index:", NOFO_INDEX_PATH)

print("NOFO index size:", nofo_index.ntotal)


Loading existing NOFO index...
NOFO index size: 11


In [18]:
# ===========================================
# Step 5F: NOFO Retrieval Helper
# ===========================================

def search_nofo(query, k=5):
    qvec = embed_query(query)
    distances, indices = nofo_index.search(qvec, k)

    results = []
    for rank, idx in enumerate(indices[0]):
        if idx == -1:
            continue

        results.append({
            "rank": rank + 1,
            "distance": float(distances[0][rank]),
            "chunk": nofo_chunks[idx]["chunk"],
            "source": "NOFO",
            "id": int(idx)
        })
    return results


In [22]:
# # ===========================================
# # Step 5G: Test NOFO Retrieval
# # ===========================================

# test_q = "review criteria"
# results = search_nofo(test_q, k=3)

# print("NOFO Retrieval Test:", test_q)
# for r in results:
#     print(f"\nRank {r['rank']}:")
#     print(r["chunk"][:400], "...")


## STEP 6 — Extract NOFO Requirements Into Structured JSON

- Retrieve each major category from the NOFO using semantic search
- Feed the retrieved NOFO text into an LLM (OpenAI GPT-4.1)
- Have the LLM extract, summarize, and structure the requirements
- Save everything into a machine-usable JSON object
- Provide a helper function to reload the JSON on demand

In [19]:
# ===============================================
# Step 6A: Retrieve NOFO Text for a Topic
# ===============================================

def retrieve_nofo_text(topic, k=5):
    """Retrieve concatenated NOFO text relevant to a topic."""
    results = search_nofo(topic, k=k)
    combined = "\n\n".join([r["chunk"] for r in results])
    return combined, results


In [20]:
# ===========================================================
# Step 6B: Extract Structured Requirements from NOFO
# ===========================================================
import re

def clean_json_output(text):
    """Remove markdown fences and extract the JSON object."""
    # Remove Markdown ```json fences
    text = text.replace("```json", "").replace("```", "").strip()

    # Extract JSON object using regex
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        return match.group(0)
    return text  # fallback


def extract_nofo_requirements(topic, k=5):
    """Use OpenAI to extract requirements from NOFO with strict JSON."""
    nofo_text, _ = retrieve_nofo_text(topic, k=k)

    prompt = f"""
You are an NIH grant reviewer. Extract ONLY structured JSON from the NOFO text.

STRICT RULES:
- RETURN ONLY A VALID JSON OBJECT.
- NO explanatory text.
- NO markdown.
- NO comments.
- NO trailing commas.

JSON FORMAT TO RETURN:
{{
  "topic": "...",
  "requirements": ["...", "..."],
  "risk_of_noncompliance": ["...", "..."],
  "citations": ["...", "..."]
}}

EXTRACT FROM NOFO TEXT BELOW:
\"\"\"{nofo_text}\"\"\"
"""

    completion = openai.chat.completions.create(
        model="gpt-4.1",
        temperature=0,
        messages=[
            {
                "role": "system", 
                "content": "You return ONLY valid JSON. No explanation. No markdown."
            },
            {"role": "user", "content": prompt}
        ]
    )

    raw = completion.choices[0].message.content.strip()

    # Try cleaning & parsing
    cleaned = clean_json_output(raw)

    try:
        data = json.loads(cleaned)
        return data
    except Exception as e:
        print("Still could not parse JSON. Saving raw content for inspection.")
        return {"topic": topic, "raw": raw}

In [21]:
# ===========================================================
# Step 6C: Build Complete NOFO Requirements Dictionary
# ===========================================================
REQ_PATH = f"{DATA_DIR}/nofo_requirements.json"

NOFO_TOPICS = [
    "Significance review criteria",
    "Innovation review criteria",
    "Approach review criteria",
    "responsiveness criteria",
    "must include",
    "prohibited activities",
    "data sharing requirements",
    "evaluation plan",
    "study design requirements",
    "digital health requirements",
    "behavioral health",
]

def build_nofo_requirements(force_rebuild=False):
    """
    Build structured NOFO requirements for all topics.
    
    - If the JSON already exists and force_rebuild=False → return cached version.
    - Otherwise → rebuild and save.
    """

    # ------------------------------------------------------------
    # 1. EARLY STOP: If requirements already built
    # ------------------------------------------------------------
    if os.path.exists(REQ_PATH) and not force_rebuild:
        print(f"NOFO requirements already exist. Loading cached file:")
        print(f"➡ {REQ_PATH}")
        with open(REQ_PATH, "r") as f:
            return json.load(f)

    # ------------------------------------------------------------
    # 2. Build Requirements Fresh
    # ------------------------------------------------------------
    print("Extracting structured NOFO requirements...\n")

    nofo_requirements = {}

    for topic in NOFO_TOPICS:
        print(f"➡ Processing topic: {topic}")

        try:
            data = extract_nofo_requirements(topic, k=6)
            nofo_requirements[topic] = data
        except Exception as e:
            print(f"Error processing '{topic}':", e)
            nofo_requirements[topic] = {"error": str(e)}

    # ------------------------------------------------------------
    # 3. Save for reuse
    # ------------------------------------------------------------
    os.makedirs(DATA_DIR, exist_ok=True)

    with open(REQ_PATH, "w") as f:
        json.dump(nofo_requirements, f, indent=2)

    print("\nSaved NOFO requirements to:", REQ_PATH)
    return nofo_requirements

nofo_requirements = build_nofo_requirements()
nofo_requirements = build_nofo_requirements()


NOFO requirements already exist. Loading cached file:
➡ ./data/nofo_requirements.json
NOFO requirements already exist. Loading cached file:
➡ ./data/nofo_requirements.json


In [22]:
# ===============================================
# Step 6D: Helper to reload NOFO requirements
# ===============================================

def load_nofo_requirements():
    with open(REQ_PATH, "r") as f:
        return json.load(f)

nofo_requirements = load_nofo_requirements()
print("📄 Loaded NOFO requirements categories:", list(nofo_requirements.keys()))


📄 Loaded NOFO requirements categories: ['Significance review criteria', 'Innovation review criteria', 'Approach review criteria', 'responsiveness criteria', 'must include', 'prohibited activities', 'data sharing requirements', 'evaluation plan', 'study design requirements', 'digital health requirements', 'behavioral health']


## STEP 7 — Build the Mistral Proposal Generator Agent

- Configure the Mistral API wrapper
- Define prompt templates for NIH proposal sections
- Build a function: generate_proposal_section()
- Integrate RAG retrieval (papers + NOFO)
- Create reference list generator
- Follow strict NIH writing style
- Keep stringency to produce short, accurate, compliant text
- Store results in ./outputs/ folder
- Prepare for LangGraph integration (Step 8 & 9)

In [23]:
# ===================================================
# Step 7A: Mistral API Wrapper
# ===================================================

import requests
import os

MISTRAL_API_KEY = os.environ["MISTRAL_API_KEY"]
MISTRAL_API_URL = "https://api.mistral.ai/v1/chat/completions"
MISTRAL_MODEL = "mistral-large-latest"

def mistral_chat(messages, temperature=0.2, max_tokens=None):
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {MISTRAL_API_KEY}"
    }

    payload = {
        "model": MISTRAL_MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    response = requests.post(MISTRAL_API_URL, json=payload, headers=headers)
    response.raise_for_status()
    data = response.json()
    return data["choices"][0]["message"]["content"]


In [24]:
# ===================================================
# Step 7B: NIH Technical Writing System Prompt
# ===================================================
NIH_SYSTEM_PROMPT = """
You are an expert NIH grant writer.

Your role:
- Generate highly technical, NIH-compliant proposal sections.
- Always follow NIH Research Strategy structure.
- Maintain strict scientific tone.
- Use concise, analytic sentences.
- Integrate NOFO requirements faithfully.
- Use only the evidence provided in RAG context.
- Cite research numerically like [1], [2].
- Never fabricate citations.

Every section you write must:
- Be directly responsive to the NOFO.
- Present a clear logic and research rationale.
- Avoid marketing, hype, or vague statements.
- Include no fluff, only scientific justification.

You DO NOT generate the reference list here.
Each section should accumulate citations that the final assembler will use to build a bibliography.
"""


In [26]:
# =========================================================================================
# Step 7C: Generate Candidate Research Ideas based on NOFO and Prior Papers
# =========================================================================================

def generate_candidate_research_ideas(nofo_requirements, top_k_papers=None):
    """
    Generate ~3-6 fundable research ideas grounded in:
    - NOFO requirements
    - Prior papers (optional: top-k most relevant)
    """
    prompt = f"""
        You are an NIH grant consultant developing research concepts
        for a digital mental health funding opportunity (NOFO).

        Use the following context:

        === NOFO REQUIREMENTS ===
        {json.dumps(nofo_requirements, indent=2)}

        === RESEARCH BACKGROUND (PAPERS) ===
        {top_k_papers if top_k_papers else "Not provided; rely on NOFO + general domain knowledge."}

        Generate 4-6 highly specific, novel, and fundable research ideas.

        For each idea include:
        - Title
        - Rationale
        - Key Methods / Data Sources
        - Expected Impact / Significance

        Produce them clearly separated with headings:
        "IDEA 1", "IDEA 2", etc.
        """

    ideas = mistral_llm.invoke(prompt).content
    return ideas

def save_candidate_ideas(ideas_text):
    save_text("./outputs/research_ideas/all_ideas.txt", ideas_text)

def pick_best_research_idea(ideas_text, nofo_requirements):
    prompt = f"""
        You are an NIH study section reviewer.

        Select the BEST research idea from the set below based on:
        - Alignment with NOFO priorities
        - Novelty
        - Feasibility
        - Significance
        - Clear digital mental health focus

        === NOFO REQUIREMENTS ===
        {json.dumps(nofo_requirements, indent=2)}

        === ALL RESEARCH IDEAS ===
        {ideas_text}

        Return ONLY:
        1. A short title
        2. A 1-paragraph summary
        3. Key methods (3-5 bullet points)
        4. Why this idea is the strongest
        """

    best = oa_llm.invoke(prompt).content  # OpenAI used because it's better evaluator
    return best


def insert_selected_idea_into_state(state, best_idea_text):
    state.selected_research_idea = best_idea_text
    return state


In [28]:
# =======================================================
# Step 7.1B: Extract bibliographic metadata from filename
# =======================================================

import re
import os

def parse_reference_from_filename(fname):
    """
    Extract best-effort metadata from a filename.
    Expected formats might include:
      Smith_2021_digital_health.pdf
      Cruickshank_Ian_mental_modeling_2019.pdf
    """

    base = os.path.splitext(fname)[0]

    # Remove underscores/dashes for parsing
    tokens = re.split(r"[_\-]", base)

    # Try to find year
    year = None
    for t in tokens:
        if re.match(r"19\d\d|20\d\d", t):
            year = t
            break

    # Build title-ish thing
    title = " ".join(tokens).replace(year if year else "", "").strip()

    return {
        "title": title.strip(),
        "year": year,
        "source_file": fname
    }


In [172]:
# =======================================================
# Step 7.1D: Assign citation numbers to RAG results
# =======================================================

def register_citation(state, citation_id, metadata=None):
    """
    Ensure this citation_id has a global citation number and metadata.
    NEVER overwrites existing metadata or numbering.
    """

    cid = str(citation_id)

    # 1. Assign number IF not assigned
    if cid not in state.citation_registry:
        state.citation_registry[cid] = len(state.citation_registry) + 1

    # 2. Add metadata ONLY if missing
    if cid not in state.citation_metadata:
        # Use provided metadata or a minimal fallback
        state.citation_metadata[cid] = metadata or {
            "title": f"Unknown Source {cid}",
            "authors": "Unknown Authors",
            "year": "n.d.",
            "venue": "",
            "doi": ""
        }

    return state.citation_registry[cid]

def assign_citations_to_results(state, research_results):
    """
    Ensures each FAISS citation_id gets a global numeric citation number.
    Returns list of (source, global_numeric_id)
    """
    assigned = []

    for r in research_results:
        cid = str(r["citation_id"])

        # Ensure global registry assignment
        num = register_citation(state, cid, metadata={
            "title":  r.get("title", ""),
            "authors": r.get("authors", ""),
            "year":    r.get("year", ""),
            "venue":   r.get("venue", ""),
            "doi":     r.get("doi", ""),
        })

        assigned.append((r["source"], num))

    return assigned



In [192]:

def build_global_citation_map_text(state):
    """
    Produces a stable list of all citations with their global numeric IDs.
    Used to prevent the LLM from resetting numbers per section.
    """
    lines = []
    # invert registry to num -> cid
    num_to_cid = {num: cid for cid, num in state.citation_registry.items()}

    for num in sorted(num_to_cid.keys()):
        cid = num_to_cid[num]
        meta = state.citation_metadata.get(cid, {})
        title = meta.get("title", f"Unknown Source {cid}")
        lines.append(f"[{num}] {title}  (cid={cid})")

    return "\n".join(lines)

In [201]:
# ===========================================================
# Step 7.1E (Updated): Proposal Section Generator
# Supports MULTIPLE NOFO topics
# ===========================================================
def generate_proposal_section(
    section_title,
    query,
    nofo_topics=None,
    k=6,
    state=None, 
    rag_chunks="",
    research_idea="", 
    max_words=None
):
    """
    Generate an NIH proposal section using:
    - RAG evidence
    - Multiple NOFO requirement topics
    - Citation-aware Mistral synthesis
    """

    # ---------------------------
    # 1. Retrieve research evidence (FAISS)
    # ---------------------------
    research_context, research_results = build_context(query, k=k)

    # Assign citation numbers to each FAISS hit
    citations = assign_citations_to_results(state, research_results) if state is not None else []

    # Build citation → source mapping text
    citation_map_text = "\n".join(
        [f"[{num}] = {src}" for src, num in citations]
    )


    # =====================================================
    # 2. Build NOFO context block
    # =====================================================
    nofo_context_block = ""

    if max_words:
        length_instruction = (
            f"WORD LIMIT: {max_words} words.\n"
            f"- Stay under {max_words} words.\n"
        )
    else:
        length_instruction = (
            "WORD LIMIT is 1500-3000. Produce full, concise NIH-style content.\n"
        )

    if nofo_topics:
        for topic in nofo_topics:
            data = nofo_requirements.get(topic)

            if not data:
                nofo_context_block += f"\n# {topic}\n(no data found)\n"
                continue

            nofo_context_block += f"\n# NOFO TOPIC: {topic}\n"

            if isinstance(data, dict) and "raw" in data:
                nofo_context_block += data["raw"] + "\n"
            else:
                if "requirements" in data:
                    nofo_context_block += "Requirements:\n" + "\n".join(
                        f"- {r}" for r in data["requirements"]
                    ) + "\n"

                if "risk_of_noncompliance" in data:
                    nofo_context_block += "Risks:\n" + "\n".join(
                        f"- {r}" for r in data["risk_of_noncompliance"]
                    ) + "\n"

                if "additional_notes" in data:
                    nofo_context_block += "Notes:\n" + "\n".join(
                        f"- {e}" for e in data["additional_notes"]
                    ) + "\n"

    # =====================================================
    # 3. Build LLM prompt
    # =====================================================
    user_prompt = f"""
        Write the **{section_title}** section of an NIH proposal.
        {length_instruction}

        === CITATION SOURCES (USE THESE NUMBERS ONLY) ===
        {build_global_citation_map_text(state)}

        === RESEARCH IDEA (Selected Strongest) ===
        {research_idea}

        === EVIDENCE (RAG Extracts) ===
        {format_chunks(rag_chunks)}

        === NOFO REQUIREMENTS ===
        {nofo_context_block}

        INSTRUCTIONS:
        - Ensure section uniquely contributes to the proposal.
        - Use numeric citations like [3], [12].
        - Do NOT hallucinate sources.
        - Use 2-5 well-structured NIH-style paragraphs.
        """

    messages = [
        {"role": "system", "content": NIH_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]

    # =====================================================
    # 4. Call LLM (Mistral)
    # =====================================================
    output = mistral_chat(messages)

    
    # Normalize to plain string text
    text = output if isinstance(output, str) else getattr(output, "content", str(output))


    return {
        "title": section_title,
        "content": text,
        "citations_used": citations,         # list of (src, cid)
        "research_results": research_results, # raw FAISS hits (optional, for debugging)
        "nofo_topics": nofo_topics,
    }



def format_chunks(chunks):
    """
    Convert a list of RAG evidence chunks into a readable block of text
    for LLM prompting.
    """
    if not chunks:
        return "No relevant prior evidence found."

    formatted = []
    for i, ch in enumerate(chunks, 1):
        text = ch.get("text", "")
        source = ch.get("source", "")
        citation = ch.get("citation_id", "")

        formatted.append(
            f"[Chunk {i}] (Source: {source}, Citation: {citation})\n{text}\n"
        )

    return "\n".join(formatted)


In [ ]:
# # ===================================================
# # Step 7.1: Test Generation
# # ===================================================

# test_section = generate_proposal_section(
#     section_title="Significance",
#     query="digital mental health outcomes",
#     nofo_topics="Significance review criteria",
#     k=4
# )

# print(test_section["content"])
# print(test_section)

{'title': 'Significance', 'content': '**Significance**\n\nThe proposed research addresses a critical and understudied gap in understanding the **neurocognitive mechanisms** by which social media engagement influences mental health outcomes, particularly in adolescents and young adults—a population at disproportionately high risk for anxiety, depression, and attentional dysregulation [3]. While epidemiological studies have established correlational links between excessive social media use and adverse psychological effects, the **neural and cognitive pathways** mediating these associations remain poorly characterized. This knowledge gap impedes the development of **targeted, mechanism-informed interventions** to mitigate harm while preserving the social and informational benefits of digital connectivity. By integrating **neuroimaging (fMRI), behavioral paradigms, and longitudinal mental health assessments**, this study will elucidate how **reward processing, attentional control, and soci

## STEP 8 — Build the LangGraph Workflow

### LangGraph Workflow Summary

This workflow automates the full lifecycle of NIH proposal generation, including:
idea generation → evidence retrieval → section drafting → iterative review →
human revision → final scoring → proposal + references.

---

### 1. Node Overview (with concise descriptions)

| Node                            | Purpose                                              |
| ------------------------------- | ---------------------------------------------------- |
| generate_research_ideas         | Mistral produces 10 ideas                            |
| select_best_idea                | Picks strongest, stores as global idea               |
| retrieve_idea_specific_evidence | RAG search specific to idea                          |
| set_next_section                | Iterates Aims → Significance → Innovation → Approach |
| generate_section                | Mistral drafts section using RAG                     |
| reviewer_evaluate_section       | GPT-4.1 provides NIH review                          |
| revise_section_after_reviewer   | Mistral improves section                             |
| combine_sections                | Assembles all accepted drafts                        |
| apply_final_human_revision      | Human edits applied                                  |
| final_reviewer_evaluation       | Full-proposal NIH scoring                            |
| finalize_proposal               | Outputs proposal + citation list                     |

### 2. Edges & Conditions (Execution Flow)

#### Idea → Evidence → First Section
generate_research_ideas
→ select_best_idea
→ node_retrieve_idea_specific_evidence
→ set_next_section

---

#### Section Loop (Draft → Review → Revise → Next)

set_next_section
├── if state.done == False → generate_section
└── if state.done == True → combine_sections


Inside the loop:

generate_section
→ reviewer_evaluate_section
→ revise_section_after_reviewer
→ set_next_section

---

#### After all sections:

combine_sections
→ apply_final_human_revision
→ final_reviewer_evaluation
→ finalize_proposal

---

### 3. Narrative Workflow Explanation

1. **Idea Generation Phase**  
   Generate many candidate ideas and select the strongest one.

2. **Evidence Retrieval Phase (RAG)**  
   Use the selected idea to pull evidence and citations.

3. **Section Drafting Loop**  
   Each NIH section is drafted, reviewed by a simulated NIH reviewer, and revised.

4. **Completion & Combination**  
   All sections are merged into a coherent draft.

5. **Human Revision**  
   User feedback is incorporated into the final draft.

6. **Final NIH Scoring**  
   The whole proposal receives an NIH-style evaluation.

7. **Finalization**  
   Final proposal text and references are output.

---

### 4. Pipeline Diagram

```
generate_research_ideas
        ↓
select_best_idea
        ↓
retrieve_idea_specific_evidence
        ↓
set_next_section ──(loop)──▶ generate_section
                                 ↓
                     reviewer_evaluate_section
                                 ↓
                     revise_section_after_reviewer
                                 ↓
                          set_next_section
        └─────────────── until all sections completed ────────────────┘
                                   ↓
                           combine_sections
                                   ↓
                        apply_final_human_revision
                                   ↓
                        final_reviewer_evaluation
                                   ↓
                            finalize_proposal
```

In [190]:
# ============================================================
# Helper: Save versions to disk
# ============================================================

import os
import json

def save_text(path, text):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

def save_json(path, data):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)


def safe_json_parse(text):
    """
    Attempts to parse the LLM output as JSON.
    If the LLM returns extra text, Markdown fences, or explanation, 
    extract the JSON block safely.
    """
    if isinstance(text, dict):
        return text  # already parsed

    if text is None:
        return {}

    # Try direct load
    try:
        return json.loads(text)
    except Exception:
        pass

    # Try to extract JSON inside code fences
    import re
    match = re.search(r"\{[\s\S]*\}", text)
    if match:
        try:
            return json.loads(match.group(0))
        except:
            pass

    # Return fallback
    return {"raw_text": text, "error": "Could not parse JSON"}

def sync_inline_citations(draft_text, state):
    ids = extract_citation_ids(draft_text)

    for cid in ids:
        cid = str(cid)

        # Never overwrite existing metadata
        if cid not in state.citation_metadata:
            state.citation_metadata[cid] = {
                "title": f"Unknown Source {cid}",
                "authors": "Unknown Authors",
                "year": "n.d.",
                "venue": "",
                "doi": ""
            }

        # Never overwrite registry numbering
        if cid not in state.citation_registry:
            state.citation_registry[cid] = len(state.citation_registry) + 1

    return draft_text



# ============================================================
# GLOBAL EMBEDDING + FAISS INITIALIZATION
# ============================================================

from sentence_transformers import SentenceTransformer
import faiss, json, numpy as np

# Embedding model
embed_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

# ---- Load FAISS index ----
faiss_index = faiss.read_index("./data/hnsw_index.faiss")

# ---- Load index→chunk mapping ----
with open("./data/paper_chunks.json", "r") as f:
    id_to_chunk = json.load(f)



In [207]:
# ============================================================
# GLOBAL: Proposal Outline
# ============================================================
PROPOSAL_OUTLINE = [
    {
        "name": "Specific Aims",
        "query": "write specific aims for this NIH proposal",
        "nofo_topics": ["goals", "expected outcomes", "significance"]
    },
    {
        "name": "Significance",
        "query": "draft the Significance section of this NIH proposal",
        "nofo_topics": ["background", "gap", "justification"]
    },
    {
        "name": "Innovation",
        "query": "draft the Innovation section",
        "nofo_topics": ["novelty", "paradigm shift"]
    },
    {
        "name": "Approach",
        "query": "draft the NIH Approach section",
        "nofo_topics": ["methodology", "design", "analyses"]
    }
]

# NIH_WORD_LIMITS = {
#     "Specific Aims": 550,          
#     "Significance": 2000,
#     "Innovation": 1200,
#     "Approach": 3400
# }

NIH_LENGTH_INSTRUCTIONS = """
NIH LENGTH REQUIREMENTS:
- Specific Aims must be ≤ 550 words (~1 page).
- Research Strategy sections (Significance, Innovation, Approach) must total ≤ 5 pages combined.
- Each section must stay near its target word limit, however please do not truncate content unnaturally.
"""


In [177]:
# =============================================================
# Step 8A: ProposalState
# =============================================================
from pydantic import BaseModel, Field
from typing import Dict, Any, List, Optional
from pydantic import field_validator

def deep_normalize_keys(obj):
    """Recursively convert all dict keys to strings."""
    if isinstance(obj, dict):
        return {str(k): deep_normalize_keys(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [deep_normalize_keys(v) for v in obj]
    else:
        return obj

class ProposalState(BaseModel):
    # ---- Core proposal content ----
    proposal_sections: Dict[str, str] = Field(default_factory=dict)
    idea_specific_chunks: List[Dict[str, Any]] = Field(default_factory=list)
    
    # source_filename → citation_number
    citation_registry: Dict[str, int] = Field(default_factory=dict)

    # citation_number → metadata dict
    citation_metadata: Dict[str, Dict[str, Any]] = Field(default_factory=dict)

    # global counter (starts at 0 — first assigned number becomes 1)
    citation_counter: int = 0

    @field_validator("citation_metadata", "citation_registry", mode="before")
    def normalize_citation_dicts(cls, v):
        if isinstance(v, dict):
            return deep_normalize_keys(v)
        return v

    # ---- Research ideas ----
    all_research_ideas: Optional[str] = None
    best_research_idea: Optional[str] = None
    best_idea_reason: Optional[str] = None
    idea_selection_details: Optional[Dict[str, Any]] = None

    # ---- Section loop state ----
    section_index: Optional[int] = None
    current_section: Optional[Dict[str, Any]] = None
    done: bool = False

    # ---- Final revision / feedback ----
    human_feedback: str = ""   # <-- REQUIRED for apply_final_human_revision
    final_reviewer_report: Optional[Dict[str, Any]] = None
    overall_feedback: Optional[str] = None           # optional but referenced by some nodes
    section_scores: Optional[Dict[str, Any]] = None  # may or may not be used depending on your scoring logic

    # ---- Final outputs ----
    reference_list: Optional[str] = None
    final_proposal_text: Optional[str] = None
    finalized_text: Optional[str] = None  # retain your old field in case other nodes write to it

    # ---- Metadata ----
    proposal_metadata: Optional[Dict[str, Any]] = None

    model_config = {"extra": "allow"}


In [ ]:
def sync_citations_from_text(state: ProposalState, text: str):
    nums = extract_citation_ids(text)

    # Build lookup from FAISS chunks
    faiss_lookup = {}
    for ch in state.idea_specific_chunks:
        cid = str(ch.get("citation_id", ""))
        if cid.isdigit():
            faiss_lookup[cid] = {
                "title": ch.get("title", "Unknown Title"),
                "authors": ch.get("authors", "Unknown Authors"),
                "year": ch.get("year", "2024"),
                "venue": ch.get("venue", ""),
                "doi": ch.get("doi", "")
            }

    for cid in nums:
        cid = str(cid)

        if cid in faiss_lookup:
            state.citation_metadata[cid] = faiss_lookup[cid]
            continue

        # fallback
        if cid not in state.citation_metadata:
            state.citation_metadata[cid] = {
                "title": f"Reference {cid}",
                "authors": "Author et al.",
                "year": "2024",
                "venue": "Journal TBD",
                "doi": ""
            }

    return state


In [42]:
# =============================================================
# Step 8B: Node - generate_research_ideas
# =============================================================

def node_generate_research_ideas(state: ProposalState):
    prompt = """
    You are an NIH grant strategist. Generate 10 innovative research ideas for a digital
    mental health R01 proposal. Each idea MUST:

    - be scientifically coherent
    - be grounded in behavioral science or digital mental health
    - be technically feasible
    - include a brief rationale

    Label them clearly:
    IDEA 1:
    IDEA 2:
    ...
    """
    response = mistral_llm.invoke(prompt)
    ideas_text = response.content

    state.all_research_ideas = ideas_text
    save_text("./outputs/research_ideas/all_ideas.txt", ideas_text)
    return state


In [44]:
# =============================================================
# Step 8B: Node - select_best_idea
# =============================================================

def node_select_best_idea(state: ProposalState):
    ideas = state.all_research_ideas

    prompt = f"""
    You are an NIH reviewer. Here are proposed research ideas:

    {ideas}

    Select the **single strongest idea** for an NIH R01 proposal.
    Provide:

    1. The strongest idea text
    2. A short explanation of WHY this idea is the strongest (innovation, feasibility, NOFO fit)

    Return in this strict JSON format:

    {{
      "best_idea": "...",
      "reason": "..."
    }}
    """

    response = openai_reviewer_llm.invoke(prompt)
    content = clean_json_text(response.content)

    try:
        result = json.loads(content)
    except:
        save_text("./outputs/research_ideas/best_idea_raw_response.txt", content)
        raise ValueError("Reviewer did not return valid JSON.")

    # Save into state correctly
    state.best_research_idea = result.get("best_idea", "")
    state.best_idea_rationale = result.get("reason", "")
    state.idea_selection_details = result

    save_json("./outputs/research_ideas/best_idea.json", result)

    return state

def clean_json_text(text: str) -> str:
    """Remove markdown fences and surrounding whitespace."""
    text = text.strip()

    # Remove ```json ... ``` or ``` ... ```
    if text.startswith("```"):
        # Remove leading ```json or ```
        first_newline = text.find("\n")
        text = text[first_newline+1:].strip()

        # Remove trailing ```
        if text.endswith("```"):
            text = text[:-3].strip()

    return text



In [164]:
def node_retrieve_idea_specific_evidence(state: ProposalState):
    print("\n🔎 Idea-specific retrieval...")

    idea_vec = embed_query(state.best_research_idea)
    D, I = index.search(idea_vec, 50)

    for idx in I[0]:
        if idx == -1: 
            continue

        chunk = metadata[idx]
        cid = str(chunk["citation_id"])

        # Always register using FAISS metadata
        register_citation(state, cid, {
            "title": chunk.get("title", f"Unknown Source {cid}"),
            "authors": chunk.get("authors", "Unknown Authors"),
            "year": chunk.get("year", "n.d."),
            "venue": chunk.get("venue", ""),
            "doi": chunk.get("doi", "")
        })

    state.citation_metadata = deep_normalize_keys(state.citation_metadata)
    return state



In [133]:
# =============================================================
# Step 8B: Node - generate_section  (with citation extraction)
# =============================================================

import re
from uuid import uuid4

def extract_citation_ids(text):
    """Return list of citation IDs like ['1','4','12'] from [1], [4], [12]"""
    return re.findall(r"\[(\d+)\]", text)


def node_generate_section(state: ProposalState):
    section = state.current_section
    name = section["name"]

    print(f"Generating initial draft for section: {name}")
    
    research_idea = state.best_research_idea or ""
    rag_context   = state.idea_specific_chunks or []

    
    # ---- LLM DRAFT GENERATION ----
    result = generate_proposal_section(
        section_title=section["name"],
        query=section["query"],
        nofo_topics=section["nofo_topics"],
        rag_chunks=rag_context,
        research_idea=research_idea, 
        max_words=None,
        state=state           # <<<<< REQUIRED
    )
    
    
    draft = result["content"]

    # ---- 2. Make sure metadata exists for all inline [n] ----
    draft = sync_inline_citations(draft, state)

    # ---- 3. Save into state ----
    state.proposal_sections[name] = draft

    # Optional: save version to disk
    save_text(f"./outputs/versions/{name}/first_draft.txt", draft)

    return state

In [134]:
# =============================================================
# Step 8C: Node - reviewer_evaluate_section
# =============================================================

def node_reviewer_evaluate_section(state: ProposalState):
    section = state.current_section
    draft = state.proposal_sections[section['name']]

    prompt = f"""
    You are an NIH scientific review officer. Evaluate the following draft section:

    SECTION: {section['name']}
    TEXT:
    {draft}

    Provide scores (1=best, 9=worst) and constructive comments.

    Return JSON:
    {{
    "score_significance": ...,
    "score_innovation": ...,
    "score_approach": ...,
    "strengths": "...",
    "weaknesses": "..."
    }}
    """

    response = openai_reviewer_llm.invoke(prompt)
    fb = json.loads(clean_json_text(response.content))

    state.proposal_sections[f"{section['name']}_reviewer_feedback"] = json.dumps(fb, indent=2)
    save_json(f"./outputs/versions/{section['name']}/reviewer_feedback.json", fb)

    return state


In [136]:
# =============================================================
# Step 8D: Node - revise_section_after_reviewer
# =============================================================
def node_revise_section_after_reviewer(state: ProposalState):
    section = state.current_section
    name = section["name"]

    original = state.proposal_sections[name]
    reviewer_fb = state.proposal_sections.get(f"{name}_reviewer_feedback", "")

    prompt = f"""
    You are revising a single NIH proposal section based on reviewer feedback.

    Rules:
    - DO NOT change citation numbers like [1], [2], [10].
    - DO NOT invent citations.
    - Maintain structure and meaning; improve clarity.

    ORIGINAL SECTION:
    {original}

    REVIEWER FEEDBACK:
    {reviewer_fb}

    Rewrite the section. Return ONLY the revised section text.
    """

    revised = mistral_llm.invoke(prompt).content.strip()
    state = sync_citations_from_text(state, revised)

    # Save revised text
    state.proposal_sections[name] = revised
    state.proposal_sections[f"{name}_revised_draft"] = revised

    save_text(f"./outputs/versions/{name}/revised_draft.txt", revised)

    return state


In [137]:
# =============================================================
# Step 8E: Node - apply_final_human_revision
# =============================================================
import re

def node_apply_final_human_revision(state: ProposalState):
    user_feedback = (state.human_feedback or "").strip()
    
    text_in = state.finalized_text   # <<< IMPORTANT

    # === BASE NIH REVISION GUIDANCE ===
    base_feedback = f"""
    You are performing a FINAL NIH-style proposal revision. 
    Revise the ENTIRE proposal according to these rules:

    1. ALIGNMENT TO RESEARCH IDEA:
    Strengthen the overall narrative so every section clearly supports 
    the core research idea and advances a unified scientific argument.

    2. CITATION ACCURACY:
    - Use ONLY citations from the provided citation map.
    - Remove any citation not present in the citation map.
    - Maintain correct numbering and prevent hallucinated references.
    - Ensure each citation is meaningfully tied to the text.
    - You MUST preserve all numeric citations exactly as written (e.g., [1], [2], [10]). Do NOT add, delete, change, or renumber any citations.


    3. REDUNDANCY & NIH-STRUCTURE:
    - Remove repetitive text across sections.
    - Ensure each section presents the shared ideas from DIFFERENT angles:
            • Significance → importance & need
            • Innovation → novelty
            • Approach → methodological execution
    - Do NOT delete essential methodological details.

    4. CLARITY & COHERENCE:
    - Improve logical flow
    - Use NIH writing style (clear, structured, technical)
    - Strengthen transitions and readability
    """

    # Merge user feedback (optional)
    if user_feedback:
        full_feedback = base_feedback + f"\n\nUSER-SPECIFIC FEEDBACK:\n{user_feedback}"
    else:
        full_feedback = base_feedback

    # Build the full proposal in section order
    full_text = ""
    for sec in PROPOSAL_OUTLINE:
        name = sec["name"]
        text = state.proposal_sections.get(name, "[Missing section]")
        full_text += f"## {name}\n{text}\n\n"

    print("Applying final NIH-aligned human revision with length, citations, and narrative checks...")

    messages = [
        {
            "role": "system",
            "content": (
                "You are a senior NIH proposal writer. Produce a polished, compliant, "
                "and concise revision of the FULL proposal. Follow all rules provided."
            )
        },
        {
            "role": "user",
            "content": f"""
            FULL PROPOSAL (to revise):
            {full_text}

            REVISION INSTRUCTIONS:
            {full_feedback}

            Return the COMPLETE revised proposal using the SAME EXACT section headers:
            {", ".join(sec["name"] for sec in PROPOSAL_OUTLINE)}.
            """
        }
    ]

    revised = mistral_chat(messages)
    revised_text = revised.content if hasattr(revised, "content") else revised
    
    revised_text = clean_llm_output(revised_text)
    state = sync_citations_from_text(state, revised_text)


    # --- Parse revised text back into sections ---
    for sec in PROPOSAL_OUTLINE:
        name = sec["name"]
        pattern = rf"##\s*{re.escape(name)}\s*(.*?)(?=##|\Z)"
        match = re.search(pattern, revised_text, re.DOTALL)

        if match:
            section_body = match.group(1).strip()

            state.proposal_sections[name] = section_body

            save_text(f"./outputs/sections/{name}/human_revised.txt", section_body)
        else:
            print(f"[Warning] Could not parse revised section: {name}")

    # Optionally re-segment sections by headings
    # (Keep your existing parser — still works)
    _apply_revised_sections_to_state(state, revised_text)
    
    # Save full revised proposal
    state.finalized_text = revised_text
    save_text("./outputs/proposal/full_human_revised_proposal.txt", revised_text)

    print("✅ Final human-guided revision applied successfully.\n")

    return state

def clean_llm_output(text: str) -> str:
    """
    Cleans the LLM output so regex section extraction always works.
    Handles:
    - leading commentary ("Here is the revised proposal")
    - Markdown fences
    - random indentation
    """
    t = text.strip()

    # Remove code fences  ```...```
    if t.startswith("```"):
        t = re.sub(r"^```.*?\n", "", t)   # remove opening fence
        t = t.rstrip("`").rstrip()

    # Remove leading "Here is..." boilerplate
    t = re.sub(r"^Here is.*?\n", "", t, flags=re.IGNORECASE)

    return t.strip()

def _apply_revised_sections_to_state(state, full_text):
    # Re-parse headings:  ## Specific Aims
    for sec in PROPOSAL_OUTLINE:
        name = sec["name"]
        pattern = rf"##\s*{re.escape(name)}\s*(.*?)(?=##|\Z)"
        match = re.search(pattern, full_text, re.DOTALL)

        if match:
            body = match.group(1).strip()
            state.proposal_sections[name] = body
        else:
            print(f"[Warning] Could not parse revised section: {name}")


In [138]:
# =============================================================
# Step 8F: Node - final_reviewer_evaluation
# =============================================================

def node_final_reviewer_evaluation(state: ProposalState):

    print("Running FINAL NIH reviewer scoring for whole proposal...")

    full_text = "\n\n".join(
        [f"## {name}\n{content}" for name, content in state.proposal_sections.items()]
    )

    system_prompt = """
    You are an NIH peer review panel. 
    You must provide structured scoring for the FULL proposal.

    Return ONLY valid JSON with the following EXACT structure:

    {
        "section_scores": {
            "Significance": <number 1-9>,
            "Innovation": <number 1-9>,
            "Approach": <number 1-9>
        },
        "overall_impact": <number 1-9>,
        "strengths": "<paragraph>",
        "weaknesses": "<paragraph>",
        "summary_statement": "<paragraph>"
    }

    Rules:
    - Scores must be integers 1-9 (1=exceptional, 9=poor)
    - All fields must appear
    - NO extra text outside JSON
    """

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": full_text}
    ]

    review_output = openai_reviewer_llm.invoke(messages)
    final_report = safe_json_parse(review_output.content)

    # Save raw report
    state.final_reviewer_report = final_report

    # Required fields for scoring-PDF generator
    state.section_scores = final_report.get("section_scores", {})
    state.overall_feedback = final_report.get("summary_statement", "")
    state.proposal_metadata = final_report.get("metadata", {})

    return state


In [159]:
# =============================================================
# Step 8G: Node - finalize_proposal
# =============================================================

def node_finalize_proposal(state: ProposalState):

    print("Finalizing proposal and preparing for PDF export...")

    # Build final proposal text
    full_text = ""
    for name, content in state.proposal_sections.items():
        full_text += f"\n\n## {name}\n{content}\n"

    # Build reference list
    references = build_reference_list(state)
    state.reference_list = references

    state.final_proposal_text = full_text + "\n\n## References\n" + references
    
    # Ensure reference list exists
    if not state.reference_list:
        state.reference_list = build_reference_list(
            state
        )

    # Ensure final text includes reference list
    if "## References" not in state.finalized_text:
        state.finalized_text += "\n\n## References\n\n" + state.reference_list



    return state

# =======================================================
# Final NIH Reference List from ProposalState
# =======================================================

def build_reference_list(state: ProposalState) -> str:
    """
    Build the reference list using:
        state.citation_registry: {citation_id → number}
        state.citation_metadata: {citation_id → metadata_dict}
    """

    if not state.citation_registry:
        return ""

    # 1. Invert registry: number → citation_id
    num_to_cid = {}
    for cid, num in state.citation_registry.items():
        num_to_cid[num] = cid   # one-to-one mapping guaranteed

    lines = []

    # 2. Iterate numerically in order
    for num in sorted(num_to_cid.keys()):
        cid = num_to_cid[num]              # citation_id string
        meta = state.citation_metadata.get(cid, {})

        authors = (meta.get("authors") or "").strip()
        year    = (str(meta.get("year")) if meta.get("year") else "").strip()
        title   = (meta.get("title") or "").strip()
        venue   = (meta.get("venue") or "").strip()
        doi     = (meta.get("doi") or "").strip()

        parts = []
        if authors:
            parts.append(authors)
        if year:
            parts.append(f"({year})")
        if title:
            if not title.endswith("."):
                title += "."
            parts.append(title)
        if venue:
            if not venue.endswith("."):
                venue += "."
            parts.append(venue)
        if doi:
            parts.append(f"doi:{doi}")

        if not parts:
            parts.append("Reference information missing.")

        lines.append(f"[{num}] " + " ".join(parts))

    return "\n".join(lines)


In [141]:
# ============================================================
# Node: set_next_section — iterate through proposal sections
# ============================================================

def node_set_next_section(state: ProposalState):

    # Initialize on very first call
    if not hasattr(state, "section_index") or state.section_index is None:
        state.section_index = 0
        state.done = False

    # If sections are complete
    if state.section_index >= len(PROPOSAL_OUTLINE):
        state.done = True
        state.current_section = None
        return state

    # Set the current section
    state.current_section = PROPOSAL_OUTLINE[state.section_index]

    # Move pointer for next cycle
    state.section_index += 1

    return state


In [81]:
# ============================================================
# Node: combine_sections — merge all drafted & revised sections
# ============================================================

def node_combine_sections(state: ProposalState):
    print("🧩 Combining all drafted + revised sections into final proposal...")

    combined = []

    for sec in PROPOSAL_OUTLINE:
        name = sec["name"]

        text = state.proposal_sections.get(
            f"{name}_revised_draft",
            state.proposal_sections.get(name, "[Missing section]")
        )

        combined.append(f"## {name}\n\n{text}\n")

    # Final combined text
    combined_text = "\n".join(combined)

    # Build references BEFORE final revision
    reference_text = build_reference_list(
        state
    )

    final_text = combined_text + "\n\n## References\n\n" + reference_text

    state.finalized_text = final_text
    state.reference_list = reference_text

    save_text("./outputs/proposal/final_proposal.txt", final_text)
    save_text("./outputs/proposal/references.txt", reference_text)

    print("✅ Combined proposal assembled.")
    return state


In [208]:
# ======================
# Step 8H: Build Graph 
# ======================
from langgraph.graph import StateGraph

workflow = StateGraph(ProposalState)

# 1. Research idea pipeline
workflow.add_node("generate_research_ideas", node_generate_research_ideas)
workflow.add_node("select_best_idea", node_select_best_idea)
workflow.add_node("node_retrieve_idea_specific_evidence", node_retrieve_idea_specific_evidence)

# 2. Section iteration controller
workflow.add_node("set_next_section", node_set_next_section)

# 3. Section drafting nodes
workflow.add_node("generate_section", node_generate_section)
workflow.add_node("reviewer_evaluate_section", node_reviewer_evaluate_section)
workflow.add_node("revise_section_after_reviewer", node_revise_section_after_reviewer)

# 4. After all sections are finished
workflow.add_node("combine_sections", node_combine_sections)

# 5. Final human revision (optional)
workflow.add_node("apply_final_human_revision", node_apply_final_human_revision)

# 6. Final reviewer evaluation (NIH full-proposal scoring)
workflow.add_node("final_reviewer_evaluation", node_final_reviewer_evaluation)

# 7. Finalize proposal
workflow.add_node("finalize_proposal", node_finalize_proposal)


## Wire up the edges
# ------------------------------------------------------------
# Research Idea → Evidence → First Section
# ------------------------------------------------------------
workflow.add_edge("generate_research_ideas", "select_best_idea")
workflow.add_edge("select_best_idea", "node_retrieve_idea_specific_evidence")
workflow.add_edge("node_retrieve_idea_specific_evidence", "set_next_section")

# ------------------------------------------------------------
# Section Loop (core drafting cycle)
# ------------------------------------------------------------
workflow.add_conditional_edges(
    "set_next_section",
    lambda state: (
        "end"
        if getattr(state, "done", False)
        else "continue"
    ),
    {
        "continue": "generate_section",
        "end": "combine_sections",
    }
)

workflow.add_edge("generate_section", "reviewer_evaluate_section")
workflow.add_edge("reviewer_evaluate_section", "revise_section_after_reviewer")
workflow.add_edge("revise_section_after_reviewer", "set_next_section")

# ------------------------------------------------------------
# After ALL sections drafted → combine → final human feedback
# ------------------------------------------------------------
workflow.add_edge("combine_sections", "apply_final_human_revision")

# After human feedback → final reviewer scoring
workflow.add_edge("apply_final_human_revision", "final_reviewer_evaluation")

# After reviewer scoring → finalize proposal text + references
workflow.add_edge("final_reviewer_evaluation", "finalize_proposal")

## Entry point
workflow.set_entry_point("generate_research_ideas")
app = workflow.compile()


In [145]:
def revise_existing_proposal_with_feedback(state, feedback: str):
    print("\n=== REVISION-ONLY PIPELINE ===")

    # 1. Inject new human feedback
    state["human_feedback"] = feedback.strip()
    print("✏️ Applying human feedback to existing finalized proposal...")

    # 2. Apply revised human update to existing final text
    state = node_apply_final_human_revision(state)

    # 3. Re-run NIH scoring on revised text
    print("🔎 Re-running final NIH scoring...")
    state = node_final_reviewer_evaluation(state)

    print("🎉 Revision complete — PDFs regenerated!")
    return state


## STEP 9 - PROPOSAL GENERATION ORCHESTRATION

- Running the full auto pipeline through LangGraph
- Saving all research ideas + selected idea
- Saving all drafts + reviewer feedback + revised drafts
- Capturing human overall revision
- Creating a human-revised full proposal
- Adding reference list to the final proposal
- Adding additional human feedback
- Applying human feedback to revise the proposal
- Generating a new version of final proposal
- Attaching reference list again


In [212]:
# ============================================================
# GLOBAL LLM INITIALIZATION (Mistral + OpenAI)
# Place this BEFORE Step 9 — ideally right after imports
# ============================================================

from langchain_openai import ChatOpenAI
from langchain_mistralai import ChatMistralAI

import json

# -----------------------
# Load API keys
# -----------------------
CONFIG_PATH = "./config.json"
with open(CONFIG_PATH) as f:
    cfg = json.load(f)

OPENAI_API_KEY = cfg["OPENAI_API_KEY"]
MISTRAL_API_KEY = cfg["MISTRAL_API_KEY"]

# -----------------------
# Initialize LLM models
# -----------------------

# Writing + Ideas (Mistral)
mistral_llm = ChatMistralAI(
    model="mistral-medium-latest",
    mistral_api_key=MISTRAL_API_KEY,
    temperature=0.25,
    max_tokens=None
)

# Reviewing + Scoring (OpenAI)
openai_reviewer_llm = ChatOpenAI(
    model="gpt-4o",
    api_key=OPENAI_API_KEY,
    temperature=0.0,
    max_tokens=4096
)

print("LLMs loaded successfully:")
print(" - Mistral (writer/ideas)")
print(" - OpenAI GPT-4o (reviewer)")


LLMs loaded successfully:
 - Mistral (writer/ideas)
 - OpenAI GPT-4o (reviewer)


In [214]:
# ============================================================
# INITIAL GENERATION: FULL PIPELINE FROM START TO END
# ============================================================

print("\n======================================")
print("🚀 FULL NIH PROPOSAL PIPELINE: STARTING EXECUTION")
print("======================================\n")
with open("./data/nofo_requirements.json", "r") as f:
    nofo_requirements = json.load(f)

# -----------------------------------------
# Create a fresh ProposalState
# -----------------------------------------
initial_state = ProposalState(
    proposal_sections={},
    citation_registry={},
    citation_metadata={},
    citation_counter= 0,
    idea_specific_chunks=[],
    human_feedback="",          # <--- EMPTY for fully automated run
    proposal_metadata={},
)
# -----------------------------------------
# FIX INITIAL STATE
# Ensure citation_metadata keys are strings
# -----------------------------------------
initial_state.citation_metadata = {
    str(k): v for k, v in initial_state.citation_metadata.items()
}

initial_state.citation_registry = {
    k: v for k, v in initial_state.citation_registry.items()
}

# -----------------------------------------
# 1. Run LangGraph workflow (all nodes)
# -----------------------------------------
print("⏳ Running LangGraph pipeline...")
state = app.invoke(initial_state, config={"recursion_limit": 200})
print("\n🎉 Pipeline Complete!\n")

# ============================================================
# SAVE INTERMEDIATE ARTIFACTS
# ============================================================

# -----------------------------------------
# Save research ideas
# -----------------------------------------
save_text("./outputs/research_ideas/all_ideas.txt", state["all_research_ideas"])
save_text("./outputs/research_ideas/best_idea.txt", state["best_research_idea"])

# -----------------------------------------
# Save drafts, reviewer feedback, revised drafts
# -----------------------------------------
for sec in PROPOSAL_OUTLINE:
    name = sec["name"]
    for suffix, ext in [
        ("first_draft", "txt"),
        ("reviewer_feedback", "json"),
        ("revised_draft", "txt")
    ]:
        key = f"{name}_{suffix}"
        if key in state["proposal_sections"]:
            path = f"./outputs/versions/{name}/{suffix}.{ext}"
            val = state["proposal_sections"][key]
            if ext == "txt":
                save_text(path, val)
            else:
                save_json(path, val)

print("✔ Saved all section drafts, feedback, and revised versions.\n")

# -----------------------------------------
# Save final full proposal output
# -----------------------------------------
save_text("./outputs/proposal/final_proposal.txt", state["finalized_text"])
save_text("./outputs/proposal/references.txt", state["reference_list"])

print("\n================ FINAL PROPOSAL ================\n")
print(state["finalized_text"])



🚀 FULL NIH PROPOSAL PIPELINE: STARTING EXECUTION

⏳ Running LangGraph pipeline...

🔎 Idea-specific retrieval...
Generating initial draft for section: Specific Aims
Generating initial draft for section: Significance
Generating initial draft for section: Innovation
Generating initial draft for section: Approach
🧩 Combining all drafted + revised sections into final proposal...
✅ Combined proposal assembled.
Applying final NIH-aligned human revision with length, citations, and narrative checks...
[Warning] Could not parse revised section: Specific Aims
[Warning] Could not parse revised section: Significance
[Warning] Could not parse revised section: Innovation
[Warning] Could not parse revised section: Approach
[Warning] Could not parse revised section: Specific Aims
[Warning] Could not parse revised section: Significance
[Warning] Could not parse revised section: Innovation
[Warning] Could not parse revised section: Approach
✅ Final human-guided revision applied successfully.

Running FI

In [210]:
if isinstance(state, dict):
    state = ProposalState(**state)
    
print(state.citation_metadata)
print(build_reference_list(state))


{'6b83d677b8f52c30037e9f6fbf55657f4b8a57e2': {'title': 'Fragile Minds: Exploring the Link Between Social Media and Young Adult Mental Health', 'authors': 'I. Mcculloh, Ben Cohen', 'year': 2023, 'venue': 'International Conference on Advances in Social Networks Analysis and Mining', 'doi': '10.1145/3625007.3627490'}, 'auto_knowing-the-terrain-explicit-and-implici': {'title': 'Knowing the terrain: explicit and implicit measures of the population', 'authors': 'McCulloh, I., & McCulloh, L.', 'year': 2018, 'venue': 'SMA White Paper What Do Others Think and How Do We Know What They Are Thinking', 'doi': ''}, 'White Paper Brain Gaze': {'title': 'Non-Invasive Synaptic Dopamine Measurement', 'authors': '', 'year': '', 'venue': '', 'doi': ''}, 'auto_a-neuro-influence-experiment-to-evaluate': {'title': 'A Neuro-Influence Experiment to Evaluate the Persuasiveness of Pre-Exposure Prophylaxis Promotion Messages Among Men Who Have Sex With Men: Protocol for a Randomized Controlled Trial', 'authors': '

In [215]:
# ============================================================
# OPTIONAL: APPLY ADDITIONAL HUMAN FEEDBACK + RE-EVALUATE + NEW PDFs
# ============================================================

print("\n======================================")
print("🔁 APPLY NEW HUMAN FEEDBACK + REGENERATE PDFs")
print("======================================\n")

# -----------------------------------------
# Add new human feedback
# -----------------------------------------
import os

# -----------------------------------------
# Ensure ProposalState object
# -----------------------------------------
if isinstance(state, dict):
    state = ProposalState(**state)

# Add new human feedback
state.human_feedback = """
Below is my full proposal draft.
Please make sure the research idea "Adaptive Digital Phenotyping for Predictive Suicide Risk Modeling" is clearly emphasized in each section.
Please revise it according to the system instructions, without removing any scientific content and without altering or renumbering any existing citations.
Your revision must:
1. Preserve All Scientific Content and Make the Proposal Concise
Keep every methodological detail, dataset description, model choice, experimental design, and ethical safeguard.
Word limit for "Specific Aims" section is 550, "Significance" section is 2000, "Innovation" section is 1200, and for "Approach" section is 3400. Do not shorten the proposal by omitting information. Make the proposal more concise.
2. Improve Writing Quality & NIH Readability
Strengthen clarity, flow, and coherence across all sections.
Ensure the narrative is unified around the core research idea.
Maintain a professional NIH tone: direct, precise, and technically grounded.
3. Remove Redundant or Repetitive Passages
If the same idea appears in multiple places, integrate it into the most appropriate section (Significance vs. Innovation vs. Approach).
Do not delete necessary scientific content—only eliminate true redundancy.
4. Preserve All Citations Exactly as Written
Do NOT add, invent, modify, delete, or renumber any citations.
Only use the citations that already exist in the draft.
Ensure each citation is properly tied to supporting statements.
5. Maintain Full Structure
Keep the overall NIH formatting (Specific Aims → Significance → Innovation → Approach).
Improve section transitions so the document reads as one cohesive scientific argument.
6. Ensure the revised proposal: matches the original research idea, reflects retrieved evidence (fNIRS, persuasion, information operations, stance classification, etc.), and avoids introducing biomedical literature I do not have citations for
Your task:
Rewrite the entire proposal into a polished, coherent, fully NIH-ready version that satisfies both the system instructions and the above revision rules.
Do not summarize.
Do not truncate.
Output the complete revised proposal.
"""

# -----------------------------------------
# Apply human revision
# -----------------------------------------
if not state.human_feedback.strip():
    print("⚠️ No human feedback provided — skipping revision.")
else:
    print("✏️ Applying your human feedback to the existing draft...")
    state = node_apply_final_human_revision(state)

# -----------------------------------------
# Rebuild reference list after human revision
# -----------------------------------------
reference_list = build_reference_list(state)

state.reference_list = reference_list

# Append references to full revised text
state.finalized_text = (
    state.finalized_text.rstrip() +
    "\n\n## References\n" +
    reference_list
)

# -----------------------------------------
# Re-run reviewer evaluation
# -----------------------------------------
print("🔎 Running new FINAL NIH reviewer evaluation...")
state = node_final_reviewer_evaluation(state)


# -----------------------------------------
# Define versioned filenames
# -----------------------------------------
def next_versioned_file(base_path, prefix):
    """
    Returns the next available filename:
    prefix_01.txt, prefix_02.txt, ...
    """
    existing = [
        f for f in os.listdir(base_path)
        if f.startswith(prefix)
    ]

    if not existing:
        version = 1
    else:
        # Extract numbers like _03
        nums = []
        for f in existing:
            import re
            m = re.search(r"_(\d+)\.", f)
            if m:
                nums.append(int(m.group(1)))
        version = (max(nums) + 1) if nums else 1

    return f"{prefix}_{version:02d}.txt"


# Create directory if missing
os.makedirs("./outputs/proposal", exist_ok=True)

# Generate versioned filenames
proposal_filename  = next_versioned_file("./outputs/proposal", "final_proposal")
scoring_filename   = next_versioned_file("./outputs/proposal", "final_scoring")
references_filename = next_versioned_file("./outputs/proposal", "references")


# -----------------------------------------
# Save files
# -----------------------------------------
save_text(f"./outputs/proposal/{proposal_filename}", state.finalized_text)
save_json(f"./outputs/proposal/{scoring_filename}", state.final_reviewer_report)
save_text(f"./outputs/proposal/{references_filename}", state.reference_list)

print(f"\n📄 Saved revised proposal as:     {proposal_filename}")
print(f"📝 Saved scoring report as:       {scoring_filename}")
print(f"📚 Saved reference list as:       {references_filename}")


# -----------------------------------------
# Print proposal in output cell
# -----------------------------------------
print("\n================ FINAL REVISED PROPOSAL ================\n")
print(state.finalized_text)
print("\n========================================================\n")
print(state.final_reviewer_report)



🔁 APPLY NEW HUMAN FEEDBACK + REGENERATE PDFs

✏️ Applying your human feedback to the existing draft...
Applying final NIH-aligned human revision with length, citations, and narrative checks...
[Warning] Could not parse revised section: Specific Aims
[Warning] Could not parse revised section: Significance
[Warning] Could not parse revised section: Innovation
[Warning] Could not parse revised section: Approach
[Warning] Could not parse revised section: Specific Aims
[Warning] Could not parse revised section: Significance
[Warning] Could not parse revised section: Innovation
[Warning] Could not parse revised section: Approach
✅ Final human-guided revision applied successfully.

🔎 Running new FINAL NIH reviewer evaluation...
Running FINAL NIH reviewer scoring for whole proposal...

📄 Saved revised proposal as:     final_proposal_01.txt
📝 Saved scoring report as:       final_scoring_01.txt
📚 Saved reference list as:       references_01.txt

================ FINAL REVISED PROPOSAL =========